# `/upload` Pipeline Walkthrough

Interactive, cell-by-cell version of what `app/routers/upload.py` does — same functions, same production code, just called directly here instead of through a FastAPI request so you can see each library's raw output before it gets wrapped into the API response.

**This notebook does not touch Azure.** It stops right before the `storage.py` calls (which need a live Storage Account connection) and instead just prints/comments what *would* happen at that point. Nothing here is uploaded or persisted anywhere — pure local exploration.

Run cells top to bottom, in order.

## Setup
Add the repo root (the repo root, one level up from this `notebooks/` folder) to `sys.path`, so `import app...` resolves the same way it does when `uvicorn` runs the real server.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

print("Notebook cwd:", Path.cwd())
print("Repo root added to sys.path:", REPO_ROOT)
# If the import in the next few cells fails with ModuleNotFoundError,
# it means your Jupyter kernel's working directory isn't this notebook's
# folder — check the "Notebook cwd" printed above and adjust REPO_ROOT.

## Step 1 — Load the raw PDF bytes
This is exactly what `await file.read()` gives the FastAPI endpoint in `routers/upload.py` — a plain `bytes` object, nothing parsed yet.

In [ ]:
SENDER_PDF_PATH = REPO_ROOT / "sample_data/sender_northbridge/Northbridge_Analytics_Sender_Offerings.pdf"

with open(SENDER_PDF_PATH, "rb") as f:
    sender_pdf_bytes = f.read()

print(f"Loaded {SENDER_PDF_PATH.name}: {len(sender_pdf_bytes):,} bytes")
print("First 8 bytes (PDF file signature):", sender_pdf_bytes[:8])

## Step 2 — pdfplumber: text extraction
Same call `parsing._extract_text_and_tables()` makes. `pdfplumber.open()` wants a file-like object, not raw bytes — `io.BytesIO(...)` just wraps our in-memory bytes so it looks like an open file, nothing written to disk.

In [ ]:
import io
import pdfplumber

with pdfplumber.open(io.BytesIO(sender_pdf_bytes)) as pdf:
    print(f"Page count: {len(pdf.pages)}")
    for i, page in enumerate(pdf.pages, start=1):
        text = page.extract_text() or ""
        print(f"\n--- Page {i} text (first 300 chars) ---")
        print(text[:300])

## Step 2b — pdfplumber: table detection
`page.extract_tables()` is a **layout heuristic** — it looks at the page's ruled lines (or aligned text gaps, if there are no lines) to infer a grid, then assigns each text token to a cell. Not OCR, not an LLM, not markdown. Works reliably here because these mock PDFs have real vector-drawn table borders (built with reportlab).

In [ ]:
with pdfplumber.open(io.BytesIO(sender_pdf_bytes)) as pdf:
    for i, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables()
        if not tables:
            print(f"Page {i}: no tables detected")
            continue
        print(f"Page {i}: found {len(tables)} table(s)")
        for table in tables:
            for row in table:
                print(row)

## Step 3 — PyMuPDF (`fitz`): image extraction
`page.get_images(full=True)` lists every embedded raster image on the page as a tuple — `img[0]` is the image's **xref**, a pointer into the PDF's internal object table, *not* the image data itself. `doc.extract_image(xref)` follows that pointer and returns the image's actual decoded bytes (real PNG/JPEG — not base64, not markdown) plus its file extension.

Images are displayed inline below via `IPython.display.Image` — nothing is saved to disk.

In [ ]:
import fitz  # PyMuPDF
from IPython.display import Image, display

with fitz.open(stream=sender_pdf_bytes, filetype="pdf") as doc:
    for page_number, page in enumerate(doc, start=1):
        image_refs = page.get_images(full=True)
        print(f"Page {page_number}: {len(image_refs)} image(s) referenced")
        for image_index, img in enumerate(image_refs, start=1):
            xref = img[0]
            base_image = doc.extract_image(xref)
            print(f"  Image {image_index}: xref={xref}, format={base_image['ext']}, "
                  f"{len(base_image['image']):,} bytes")
            display(Image(data=base_image["image"]))

## Step 4 — now call the real production function
Everything above was done by hand to show the mechanics. `parsing.parse_pdf()` is the actual function `routers/upload.py` calls — confirm it produces the same page/table/image counts as the manual steps above.

In [ ]:
from app.parsing import parse_pdf

sender_parsed = parse_pdf(sender_pdf_bytes, SENDER_PDF_PATH.name)

print("type:", type(sender_parsed))  # a Pydantic ParsedDocument, not a dict
print("page_count:", sender_parsed.page_count)
print("tables found:", len(sender_parsed.tables))
print("images found:", len(sender_parsed.images))
print("text length:", len(sender_parsed.text), "chars")
print("\nFirst table:")
for row in (sender_parsed.tables[0] if sender_parsed.tables else []):
    print(" ", row)

## Step 5 — this is a *validated* Pydantic object, not a plain dict
Because `ParsedDocument` is a Pydantic model, every field was checked against its declared type the moment `parse_pdf()` constructed it — if parsing had produced the wrong shape anywhere, this would already have raised a `ValidationError` instead of letting bad data flow downstream. `.model_dump()` gives back a plain dict view of the same, now-validated data (image bytes replaced with a length so it's actually readable here).

In [ ]:
summary = sender_parsed.model_dump()
summary["images"] = [
    {"filename": img["filename"], "page_number": img["page_number"], "content_bytes": len(img["content"])}
    for img in summary["images"]
]
summary["text"] = summary["text"][:200] + "..."  # full text is long; just previewing here
summary

## Step 6 — what happens next in the real endpoint (NOT run here)
In `routers/upload.py`, this `parsed` object plus the raw bytes get handed to `storage.py`, which needs a live Azure Storage connection — skipped in this notebook. For reference, this is the blob path it would use (see `CODE_MAP.md` for the full picture):

In [ ]:
from uuid import uuid4

document_id = uuid4()
sender_id = "northbridge-analytics"
receiver_id = "ferrow-industrial"
role = "sender"

example_blob_path = f"{sender_id}/{receiver_id}/{role}/{document_id}/{SENDER_PDF_PATH.name}"
print("storage.upload_raw_pdf() would use this Blob path (not actually uploaded here):")
print(example_blob_path)
print("\n...and storage.save_document_metadata() would write one Table row with:")
print(f"  PartitionKey = {sender_id}__{receiver_id}")
print(f"  RowKey       = {document_id}")

---
## Now the RECEIVER document (Ferrow)
Same production function, different file — mechanics already shown above, so this just runs `parse_pdf()` directly and shows the result.

In [ ]:
RECEIVER_PDF_PATH = REPO_ROOT / "sample_data/receiver_ferrow/Ferrow_Industrial_Group_Receiver_Brief.pdf"

with open(RECEIVER_PDF_PATH, "rb") as f:
    receiver_pdf_bytes = f.read()

receiver_parsed = parse_pdf(receiver_pdf_bytes, RECEIVER_PDF_PATH.name)

print(f"Loaded {RECEIVER_PDF_PATH.name}: {len(receiver_pdf_bytes):,} bytes\n")
print("page_count:", receiver_parsed.page_count)
print("tables found:", len(receiver_parsed.tables))
print("images found:", len(receiver_parsed.images))
print("text length:", len(receiver_parsed.text), "chars")
print("\nFirst table:")
for row in (receiver_parsed.tables[0] if receiver_parsed.tables else []):
    print(" ", row)

Inline preview of the receiver's extracted image(s) too, same as Step 3:

In [ ]:
with fitz.open(stream=receiver_pdf_bytes, filetype="pdf") as doc:
    for page_number, page in enumerate(doc, start=1):
        for img in page.get_images(full=True):
            base_image = doc.extract_image(img[0])
            display(Image(data=base_image["image"]))

## End — summary of both documents
Sanity-check both parses side by side before moving on.

In [ ]:
for name, parsed in [("Sender (Northbridge)", sender_parsed), ("Receiver (Ferrow)", receiver_parsed)]:
    print(f"{name}: {parsed.page_count} pages, {len(parsed.tables)} table(s), "
          f"{len(parsed.images)} image(s), {len(parsed.text)} chars of text")